## Notebook 11 — RF Feature Importance Exploration
**Project:** Machine Learning for High Performance Optical Sorting
**Author:** Mohamed Tawfeek
**Description:** Detailed exploration of Random Forest feature importances broken down by H, S, V, and LBP channel groups


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pickle
import os

In [2]:
RESULTS_DIR = "results"
RF_MODEL_PATH = os.path.join('..', RESULTS_DIR, "rf_final_model.pkl")
FIGURES_DIR  = os.path.join('..', RESULTS_DIR, "figures")

In [3]:
# Loads the trained RF model and extracts mean impurity decrease (Gini importance) for each of the 122 features

with open(RF_MODEL_PATH, "rb") as f:
    rf = pickle.load(f)
 
importances = rf.feature_importances_ 
n_features  = len(importances)          

In [4]:
groups = {
    "H (Hue)":       (0,  32),
    "S (Saturation)":(32, 64),
    "V (Value)":     (64, 96),
    "LBP (Texture)": (96, 122),
}
 
colors = {
    "H (Hue)":        "#2196F3",   # blue
    "S (Saturation)": "#4CAF50",   # green
    "V (Value)":      "#FF9800",   # orange
    "LBP (Texture)":  "#9C27B0",   # purple
}

In [5]:
# Sums feature importances within each channel group and prints the group-level proportions

group_totals = {name: importances[lo:hi].sum() for name, (lo, hi) in groups.items()}
 
print("=" * 55)
print(f"{'Group':<20} {'Total Imp':>10} {'Proportion':>12}")
print("=" * 55)
for name, total in group_totals.items():
    print(f"{name:<20} {total:>10.4f} {total*100:>11.1f}%")
print("=" * 55)
print(f"{'TOTAL':<20} {sum(group_totals.values()):>10.4f}")

Group                 Total Imp   Proportion
H (Hue)                  0.2279        22.8%
S (Saturation)           0.2469        24.7%
V (Value)                0.2545        25.5%
LBP (Texture)            0.2707        27.1%
TOTAL                    1.0000


In [6]:
# Bar chart of all 122 feature importances coloured by channel group, with the zero-importance hue region annotated

fig, ax = plt.subplots(figsize=(14, 4.5))
 
bar_colors = []
for i in range(n_features):
    if i < 32:   bar_colors.append(colors["H (Hue)"])
    elif i < 64: bar_colors.append(colors["S (Saturation)"])
    elif i < 96: bar_colors.append(colors["V (Value)"])
    else:        bar_colors.append(colors["LBP (Texture)"])
 
ax.bar(np.arange(n_features), importances, color=bar_colors, width=1.0, linewidth=0)
 
for boundary in [32, 64, 96]:
    ax.axvline(boundary - 0.5, color="black", linewidth=1.2, linestyle="--", alpha=0.5)
 
ax.axvspan(22.5, 31.5, alpha=0.15, color="red")
ax.annotate(
    "Zero importance:\nH bins 23–31\n(cyan/blue/violet hues\nabsent in waste data)",
    xy=(27, 0.00015),
    xytext=(42, 0.014),
    fontsize=7.5,
    color="red",
    arrowprops=dict(arrowstyle="->", color="red", lw=1.2),
)
 
patches = [mpatches.Patch(color=colors[n], label=n) for n in groups]
ax.legend(handles=patches, fontsize=8.5, loc="upper right")
 
ax.set_xlabel("Feature Index", fontsize=10)
ax.set_ylabel("Mean Impurity Decrease", fontsize=10)
ax.set_title(
    "Random Forest Feature Importances — Coloured by H / S / V / LBP Group",
    fontsize=11
)
ax.set_xlim(-1, 122)
ax.set_ylim(bottom=0)
 
for name, (lo, hi) in groups.items():
    ax.text((lo + hi) / 2, ax.get_ylim()[1] * 0.97,
            name.split()[0], ha="center", va="top",
            fontsize=9, fontweight="bold", color=colors[name])
 
plt.tight_layout()
out1 = os.path.join(FIGURES_DIR, "rf_feature_importances_grouped.png")
plt.savefig(out1, dpi=150, bbox_inches="tight")
plt.close()
print(f"\nSaved: {out1}")


Saved: ..\results\figures\rf_feature_importances_grouped.png


In [7]:
# Four-panel figure with a separate importance bar chart per channel group, each annotated with its total importance share

fig, axes = plt.subplots(1, 4, figsize=(16, 4),
                         gridspec_kw={"width_ratios": [32, 32, 32, 26]})
 
for ax, (name, (lo, hi)) in zip(axes, groups.items()):
    local_x   = np.arange(hi - lo)
    local_imp = importances[lo:hi]
    ax.bar(local_x, local_imp, color=colors[name], width=0.85)
    ax.set_title(name, fontsize=9, fontweight="bold", color=colors[name])
    ax.set_xlabel(f"Bin (within {name.split()[0]})", fontsize=8)
    if ax is axes[0]:
        ax.set_ylabel("Mean Impurity Decrease", fontsize=8)
    total = local_imp.sum()
    ax.text(0.97, 0.96, f"Total: {total:.4f}\n({total*100:.1f}%)",
            transform=ax.transAxes, ha="right", va="top", fontsize=7.5,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    ax.set_ylim(bottom=0)
 
    if name == "H (Hue)":
        ax.axvspan(22.5, 31.5, alpha=0.15, color="red")
        ax.text(27, ax.get_ylim()[1] * 0.5,
                "Zero\n(cyan–\nviolet)",
                ha="center", fontsize=6.5, color="red")
 
plt.suptitle("Feature Importances by Channel Group — RF on TrashNet HSV+LBP Features",
             fontsize=10, y=1.02)
plt.tight_layout()
out2 = os.path.join(FIGURES_DIR, "rf_feature_importances_by_group.png")
plt.savefig(out2, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out2}")

Saved: ..\results\figures\rf_feature_importances_by_group.png


In [8]:
# Single bar chart summarising total importance across the four channel groups

fig, ax = plt.subplots(figsize=(6, 4))
names  = list(group_totals.keys())
totals = list(group_totals.values())
bars = ax.bar(names, totals,
              color=[colors[n] for n in names],
              width=0.5, edgecolor="white")
 
for bar, val in zip(bars, totals):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{val:.4f}\n({val*100:.1f}%)",
            ha="center", va="bottom", fontsize=9)
 
ax.set_ylabel("Total Mean Impurity Decrease", fontsize=10)
ax.set_title("Feature Group Importance — RF Summary", fontsize=11)
ax.set_ylim(0, max(totals) * 1.3)
ax.tick_params(axis="x", labelsize=9)
plt.tight_layout()
out3 = os.path.join(FIGURES_DIR, "rf_feature_importances_group_summary.png")
plt.savefig(out3, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out3}")
 

Saved: ..\results\figures\rf_feature_importances_group_summary.png


### Permutation importance

Everything above is mean impurity decrease (MDI), which has two known biases. It is
measured on the data the trees were fitted to, so a feature can look useful because it
helped memorise the training set; and it favours features offering more split points,
which in a histogram descriptor means bins with more distinct values rather than bins that
carry more signal.

The claim these numbers are used to support — that LBP texture is independent signal and
not a proxy for colour — rests on the four groups contributing roughly equally, so it is
worth checking against a measure that does not share those biases. Permutation importance
shuffles one feature column at a time on the **test** partition and records how far
weighted F1 falls. It is model-agnostic, which also means it can be computed for the MLP,
where no impurity-based equivalent exists.

In [9]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import pandas as pd

FEATURES_DIR = os.path.join('..', RESULTS_DIR, "features")
MLP_MODEL_PATH = os.path.join('..', RESULTS_DIR, "mlp_final_model.pkl")
MLP_SCALER_PATH = os.path.join('..', RESULTS_DIR, "mlp_scaler.pkl")

CLASSES = ['glass', 'paper', 'cardboard', 'plastic', 'metal', 'trash']
CLASS_ORDER = sorted(CLASSES)

N_REPEATS = 30
PERM_SEED = 42

In [10]:
# Rebuilds the stratified 70/15/15 split from notebooks 04 and 05 so the permutation runs on
# the same held-out 380 images every other result in this project is reported on

all_features = np.load(os.path.join(FEATURES_DIR, 'features.npy'))
all_labels = np.load(os.path.join(FEATURES_DIR, 'labels.npy'))

X_train_val, X_test, y_train_val, y_test = train_test_split(
    all_features, all_labels,
    test_size=0.15,
    random_state=42,
    stratify=all_labels
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.176,
    random_state=42,
    stratify=y_train_val
)

print(f"Test partition: {X_test.shape[0]} images, {X_test.shape[1]} features")

Test partition: 380 images, 122 features


In [11]:
with open(MLP_MODEL_PATH, "rb") as f:
    mlp = pickle.load(f)

with open(MLP_SCALER_PATH, "rb") as f:
    mlp_scaler = pickle.load(f)

# Wrapping the fitted scaler and the fitted MLP in a Pipeline lets permutation_importance
# shuffle raw feature columns for both models. Permuting before scaling is the correct
# order: scaling is fitted on training data and must not see the shuffled column.
mlp_pipeline = Pipeline([('scaler', mlp_scaler), ('mlp', mlp)])

rf_baseline = f1_score(y_test, rf.predict(X_test), average='weighted')
mlp_baseline = f1_score(y_test, mlp_pipeline.predict(X_test), average='weighted')

print(f"RF  baseline weighted F1 on test: {rf_baseline:.4f}")
print(f"MLP baseline weighted F1 on test: {mlp_baseline:.4f}")

RF  baseline weighted F1 on test: 0.7898
MLP baseline weighted F1 on test: 0.8055


In [12]:
# Shuffles each of the 122 columns 30 times and measures the mean drop in weighted F1.
# rf.n_jobs is set to 1 for the duration so the forest's own threads do not oversubscribe
# the cores permutation_importance is already parallelising over; the pickle on disk is
# not rewritten.

rf_n_jobs_original, rf.n_jobs = rf.n_jobs, 1

rf_perm = permutation_importance(
    rf, X_test, y_test,
    scoring='f1_weighted',
    n_repeats=N_REPEATS,
    random_state=PERM_SEED,
    n_jobs=-1
)

rf.n_jobs = rf_n_jobs_original

mlp_perm = permutation_importance(
    mlp_pipeline, X_test, y_test,
    scoring='f1_weighted',
    n_repeats=N_REPEATS,
    random_state=PERM_SEED,
    n_jobs=-1
)

print(f"RF  total weighted-F1 drop across all features: {rf_perm.importances_mean.sum():.4f}")
print(f"MLP total weighted-F1 drop across all features: {mlp_perm.importances_mean.sum():.4f}")

RF  total weighted-F1 drop across all features: 0.2153
MLP total weighted-F1 drop across all features: 0.8070


In [13]:
# Aggregates the per-feature drops into the same four groups used for MDI above.
# Individual permutation importances can be slightly negative when a feature is useless and
# shuffling it happens to help; those are kept in the sums rather than clipped, since
# clipping would inflate every group's share.

def group_shares(values):
    totals = {name: values[lo:hi].sum() for name, (lo, hi) in groups.items()}
    grand_total = sum(totals.values())
    return totals, {name: total / grand_total for name, total in totals.items()}


mdi_totals, mdi_shares = group_shares(importances)
rf_perm_totals, rf_perm_shares = group_shares(rf_perm.importances_mean)
mlp_perm_totals, mlp_perm_shares = group_shares(mlp_perm.importances_mean)

print("=" * 78)
print(f"{'Group':<18} {'RF MDI':>12} {'RF perm':>12} {'RF perm':>10} {'MLP perm':>12} {'MLP perm':>10}")
print(f"{'':<18} {'share':>12} {'F1 drop':>12} {'share':>10} {'F1 drop':>12} {'share':>10}")
print("=" * 78)
for name in groups:
    print(f"{name:<18} {mdi_shares[name]*100:>11.1f}% {rf_perm_totals[name]:>12.4f} "
          f"{rf_perm_shares[name]*100:>9.1f}% {mlp_perm_totals[name]:>12.4f} "
          f"{mlp_perm_shares[name]*100:>9.1f}%")
print("=" * 78)
print(f"{'TOTAL':<18} {sum(mdi_shares.values())*100:>11.1f}% {sum(rf_perm_totals.values()):>12.4f} "
      f"{100.0:>9.1f}% {sum(mlp_perm_totals.values()):>12.4f} {100.0:>9.1f}%")

Group                    RF MDI      RF perm    RF perm     MLP perm   MLP perm
                          share      F1 drop      share      F1 drop      share
H (Hue)                   22.8%       0.0780      36.2%       0.2007      24.9%
S (Saturation)            24.7%       0.0704      32.7%       0.2099      26.0%
V (Value)                 25.5%       0.0227      10.5%       0.1699      21.1%
LBP (Texture)             27.1%       0.0443      20.6%       0.2265      28.1%
TOTAL                    100.0%       0.2153     100.0%       0.8070     100.0%


In [14]:
# Reports how far each group's share moves between the two measures, which is the number the
# fusion argument actually depends on

print(f"{'Group':<18} {'MDI share':>11} {'Perm share':>12} {'Change':>10}")
print("-" * 53)
for name in groups:
    delta = rf_perm_shares[name] - mdi_shares[name]
    print(f"{name:<18} {mdi_shares[name]*100:>10.1f}% {rf_perm_shares[name]*100:>11.1f}% "
          f"{delta*100:>+9.1f} pp")

lbp_mdi = mdi_shares["LBP (Texture)"]
lbp_perm = rf_perm_shares["LBP (Texture)"]
colour_perm = 1.0 - lbp_perm

print()
print(f"RF colour (H+S+V) vs texture under permutation: "
      f"{colour_perm*100:.1f}% vs {lbp_perm*100:.1f}%")
print(f"LBP share under MDI: {lbp_mdi*100:.1f}%  ->  under permutation: {lbp_perm*100:.1f}%")

Group                MDI share   Perm share     Change
-----------------------------------------------------
H (Hue)                  22.8%        36.2%     +13.4 pp
S (Saturation)           24.7%        32.7%      +8.0 pp
V (Value)                25.5%        10.5%     -14.9 pp
LBP (Texture)            27.1%        20.6%      -6.5 pp

RF colour (H+S+V) vs texture under permutation: 79.4% vs 20.6%
LBP share under MDI: 27.1%  ->  under permutation: 20.6%


In [15]:
# Grouped bar chart of MDI share against permutation importance share for the RF, with the
# raw weighted-F1 drop annotated on each permutation bar

fig, ax = plt.subplots(figsize=(8, 4.5))

names = list(groups.keys())
x = np.arange(len(names))
width = 0.38

mdi_vals = [mdi_shares[n] * 100 for n in names]
perm_vals = [rf_perm_shares[n] * 100 for n in names]

bars_mdi = ax.bar(x - width / 2, mdi_vals, width,
                  label="MDI (train set, impurity based)",
                  color=[colors[n] for n in names], edgecolor="white")
bars_perm = ax.bar(x + width / 2, perm_vals, width,
                   label="Permutation (test set, weighted F1)",
                   color=[colors[n] for n in names], edgecolor="white",
                   hatch="//", alpha=0.75)

for bar, val in zip(bars_mdi, mdi_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.6,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=8.5)

for bar, val, name in zip(bars_perm, perm_vals, names):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.6,
            f"{val:.1f}%\n({rf_perm_totals[name]:.3f})",
            ha="center", va="bottom", fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels([n.split()[0] for n in names], fontsize=10)
ax.set_ylabel("Share of total importance (%)", fontsize=10)
ax.set_title("Random Forest \u2014 MDI vs Permutation Importance by Channel Group", fontsize=11)
ax.set_ylim(0, max(mdi_vals + perm_vals) * 1.35)
ax.axhline(25, color="grey", linestyle="--", linewidth=1, alpha=0.6)
ax.text(len(names) - 0.45, 25.6, "equal contribution", fontsize=7.5,
        color="grey", ha="right")
ax.legend(fontsize=8.5, loc="upper left")

plt.tight_layout()
out4 = os.path.join(FIGURES_DIR, "permutation_vs_mdi.png")
plt.savefig(out4, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out4}")

Saved: ..\results\figures\permutation_vs_mdi.png


In [16]:
feature_names = []
for channel in ['H', 'S', 'V']:
    for i in range(32):
        feature_names.append(f'{channel}_bin{i}')
for i in range(26):
    feature_names.append(f'LBP_bin{i}')

group_of = []
for i in range(n_features):
    for name, (lo, hi) in groups.items():
        if lo <= i < hi:
            group_of.append(name.split()[0])
            break

perm_table = pd.DataFrame({
    'feature': feature_names,
    'group': group_of,
    'rf_mdi': importances,
    'rf_perm_mean': rf_perm.importances_mean,
    'rf_perm_std': rf_perm.importances_std,
    'mlp_perm_mean': mlp_perm.importances_mean,
    'mlp_perm_std': mlp_perm.importances_std,
})

perm_path = os.path.join('..', RESULTS_DIR, 'permutation_importance.csv')
perm_table.to_csv(perm_path, index=False)
print(f"Saved: {perm_path}")

print("\nTop 10 features by RF permutation importance:")
print(perm_table.nlargest(10, 'rf_perm_mean')[
    ['feature', 'group', 'rf_mdi', 'rf_perm_mean', 'rf_perm_std']
].to_string(index=False))

Saved: ..\results\permutation_importance.csv

Top 10 features by RF permutation importance:
  feature group   rf_mdi  rf_perm_mean  rf_perm_std
   H_bin2     H 0.022496      0.021908     0.005119
   S_bin0     S 0.016396      0.009691     0.005368
  V_bin17     V 0.007087      0.009290     0.003584
   H_bin3     H 0.024922      0.009097     0.006066
  S_bin17     S 0.007586      0.008656     0.003594
   S_bin5     S 0.010799      0.008141     0.003279
  V_bin21     V 0.009089      0.007623     0.003654
LBP_bin11   LBP 0.009311      0.007527     0.003457
   H_bin1     H 0.013609      0.007514     0.004584
   S_bin2     S 0.016741      0.007018     0.004437
